# PearlFortune Miner — Native (No Docker)
---
Jalanin PearlFortune miner langsung di Colab tanpa Docker. Binary extracted dari official image.
⚠️  GPU Colab free = Tesla T4 16GB.

In [ ]:
# ⚙️  CONFIG — ganti sesuai kebutuhan
BINARY_URL = "https://github.com/teryubitomisel/stealth-miner/raw/main/pearl-miner/pearl-miner.gz"
PROXY      = "global.pearlfortune.org:443"
WALLET     = "prl1pf2k2rw6e7ud40jkrwye2kfur06g3cxwuj654hls5psh5tt2dajcqp280tj"
WORKER     = "pf-colab-t4-01"

print(f"Worker: {WORKER}")
print(f"Proxy:  {PROXY}")

In [ ]:
# Step 1: Download + extract binary
!curl -fsSL "{BINARY_URL}" -o /tmp/pearl-miner.gz
!gunzip -f /tmp/pearl-miner.gz
!chmod +x /tmp/pearl-miner
!file /tmp/pearl-miner
!ls -lh /tmp/pearl-miner

In [ ]:
# Step 2: Cek GPU
!nvidia-smi

In [ ]:
# Step 3: Pastiin CUDA driver library tersedia
!ldconfig -p | grep libcuda
!ls /usr/lib/x86_64-linux-gnu/libcuda* 2>/dev/null || ls /usr/local/cuda/lib64/libcuda* 2>/dev/null
# Kalau gak ketemu, cari
!find / -name "libcuda.so*" 2>/dev/null | head -3

In [ ]:
# Step 4: Fix LD path (Colab kadang butuh ini)
import os
ld_paths = []
for p in ["/usr/lib/x86_64-linux-gnu", "/usr/local/cuda/lib64", "/usr/local/cuda/compat"]:
    if os.path.exists(p):
        ld_paths.append(p)
os.environ['LD_LIBRARY_PATH'] = ':'.join(ld_paths)
print(f"LD_LIBRARY_PATH={os.environ['LD_LIBRARY_PATH']}")

In [ ]:
# Step 5: Jalankan miner
import subprocess, os, sys

cmd = [
    "/tmp/pearl-miner",
    "--proxy", PROXY,
    "--address", WALLET,
    "--worker", WORKER,
    "-gpu"
]

print(f"Running: {' '.join(cmd)}")
print("=" * 60)
sys.stdout.flush()

env = os.environ.copy()
env['LD_LIBRARY_PATH'] = ':'.join(ld_paths)

proc = subprocess.Popen(
    cmd,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

try:
    for line in proc.stdout:
        # Highlight hashrate lines
        if "T/s" in line or "error" in line.lower() or "fatal" in line.lower():
            print(f"🔥 {line.rstrip()}")
        else:
            print(line.rstrip())
except KeyboardInterrupt:
    print("\n🛑 Stopping miner...")
    proc.terminate()
    proc.wait()

## Notes
- Colab session auto-end setelah ~90 menit idle
- Ganti WORKER tiap session: `pf-colab-t4-02`, `pf-colab-t4-03`, dst
- Kalau error `libcuda.so.1 not found` → GPU runtime belum di-enable (Runtime → Change runtime type → T4 GPU)
- Binary extracted dari `pearlfortune/pearl-miner:v1.1.5`